# SmallCall — Stage 1 (v2): Data + Baseline

**Goal of this stage:** measure how good a small model (Qwen2.5-1.5B-Instruct) and a much larger model (gpt-oss-120b, a 120B open-weight model served by Groq) are at tool calling *before* any training. Every later stage is compared against these numbers.

**What is tested**
- **Unseen tools:** every test question needs a tool that never appears in the training data. This measures whether the model can use a new API from its description, not whether it memorised one.
- **No suitable tool:** the right tool is removed from the list, so the correct behaviour is to *not* call anything.

**What this notebook does**
1. Loads the xLAM function-calling dataset and builds train / val / test splits (saved to Drive).
2. Runs the small model on the 550 test examples and scores each answer by code.
3. Runs the large model on 200 of the same examples (optional, needs a Groq key).
4. Prints a comparison table and groups the failures into categories.

**Before you run:** Runtime → Change runtime type → **GPU** (L4 best; T4 also works).

Everything is saved to `MyDrive/smallcall/`. If Colab disconnects, run all cells again — finished work is skipped.

## 1. Check the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


## 2. Install libraries

In [2]:
%pip install -q -U "transformers>=4.46" "datasets>=3.0" accelerate openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 147.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 103.7 MB/s eta 0:00:00


## 3. Connect Google Drive and set config

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random, re, time

PROJECT_DIR = "/content/drive/MyDrive/smallcall"
DATA_DIR    = f"{PROJECT_DIR}/data_v2"
RESULTS_DIR = f"{PROJECT_DIR}/results_v2"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_TOOLS = 6            # skip examples offering more tools (keeps prompts short)
HELD_OUT_FRAC = 0.15     # share of tools that are kept out of training completely

# (examples that need a call, examples where no tool fits)
N_TEST  = (400, 150)     # test questions use held-out tools only
N_VAL   = (270, 30)
N_TRAIN = (7200, 800)    # used in Stage 2

BIG_MODEL = "openai/gpt-oss-120b"   # 120B open-weight model served by Groq
N_BIG = (150, 50)                   # big-model baseline runs on this many test examples

def write_jsonl(path, rows):
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

def read_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

print("Saving everything under", PROJECT_DIR)

Mounted at /content/drive
Saving everything under /content/drive/MyDrive/smallcall


## 4. Log in (Hugging Face + Groq)

Uses Colab **Secrets** (the key icon in the left sidebar). You need:
- `HF_TOKEN` — required (the dataset is gated)
- `GROQ_API_KEY` — optional; without it the big-model baseline is skipped

In [4]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None
    print("No GROQ_API_KEY found - big-model baseline will be skipped.")

## 5. Convert xLAM tool definitions to standard JSON-schema tools

xLAM describes arguments with Python-style types (`"str"`, `"List[int]"`, `"float, optional"`). Both Qwen's chat template and the Groq API expect standard JSON-schema tools, so we convert once here.

In [5]:
TYPE_MAP = {
    "str": "string", "string": "string",
    "int": "integer", "integer": "integer",
    "float": "number", "number": "number",
    "bool": "boolean", "boolean": "boolean",
    "list": "array", "tuple": "array", "set": "array", "array": "array",
    "dict": "object", "object": "object",
}

def to_json_schema_type(raw_type):
    # 'float, optional' -> float ; 'List[int]' -> array of integer
    base = str(raw_type or "str").split(",")[0].strip()
    low = base.lower().replace("typing.", "")
    m = re.match(r"^(list|tuple|set)\[(.+)\]$", low)
    if m:
        inner = to_json_schema_type(m.group(2))
        return {"type": "array", "items": {"type": inner["type"]}}
    if low.startswith(("list", "tuple", "set")):
        return {"type": "array", "items": {"type": "string"}}
    if low.startswith("dict"):
        return {"type": "object"}
    return {"type": TYPE_MAP.get(low, "string")}

def convert_tool(tool):
    params = tool.get("parameters") or {}
    # Already JSON-schema style? keep as is.
    if params.get("type") == "object" and "properties" in params:
        schema = params
    else:
        props, required = {}, []
        for pname, spec in params.items():
            spec = spec or {}
            raw_type = str(spec.get("type", "str"))
            prop = to_json_schema_type(raw_type)
            desc = spec.get("description", "")
            if spec.get("default") not in (None, ""):
                desc += f" (default: {spec['default']})"
            prop["description"] = desc
            props[pname] = prop
            is_optional = "optional" in raw_type.lower() or "default" in spec
            is_required = bool(spec["required"]) if "required" in spec else not is_optional
            if is_required:
                required.append(pname)
        schema = {"type": "object", "properties": props, "required": required}
    return {"type": "function",
            "function": {"name": tool["name"],
                         "description": tool.get("description", ""),
                         "parameters": schema}}

## 6. Load xLAM and build the splits (saved to Drive, made only once)

- 15% of all tools are **held out**. Training examples never offer a held-out tool; every test question needs one.
- **No-suitable-tool** examples are made by deleting the correct tool(s) from an example's tool list. The correct answer is then "no call".

In [6]:
from datasets import load_dataset

def as_obj(x):
    return x if isinstance(x, (list, dict)) else json.loads(x)

def tool_names(ex):
    return {t["function"]["name"] for t in ex["tools"]}

def gold_names(ex):
    return {a["name"] for a in ex["answers"]}

def make_no_call(ex):
    # Remove the correct tool(s); what is left cannot answer the question.
    remaining = [t for t in ex["tools"] if t["function"]["name"] not in gold_names(ex)]
    if not remaining:
        return None
    return {**ex, "id": ex["id"] + "-nocall", "tools": remaining, "answers": [], "kind": "no_call"}

def take(pool, n_call, n_no_call):
    # Build n_no_call "no suitable tool" examples first, then take n_call normal examples.
    no_calls, rest = [], []
    for ex in pool:
        nc = make_no_call(ex) if len(no_calls) < n_no_call else None
        if nc:
            no_calls.append(nc)
        else:
            rest.append(ex)
    return rest[:n_call] + no_calls, rest[n_call:]

paths = {s: f"{DATA_DIR}/{s}.jsonl" for s in ["train", "val", "test"]}

if all(os.path.exists(p) for p in paths.values()):
    splits = {s: read_jsonl(p) for s, p in paths.items()}
    print("Loaded existing splits from Drive.")
else:
    ds = load_dataset("Salesforce/xlam-function-calling-60k", split="train")
    rows, skipped = [], 0
    for i, r in enumerate(ds):
        try:
            tools, answers = as_obj(r["tools"]), as_obj(r["answers"])
            if not tools or not answers or len(tools) > MAX_TOOLS:
                skipped += 1
                continue
            ex = {"id": f"xlam-{r.get('id', i)}",
                  "query": r["query"],
                  "tools": [convert_tool(t) for t in tools],
                  "answers": [{"name": a["name"], "arguments": a.get("arguments") or {}} for a in answers],
                  "kind": "call"}
            if not gold_names(ex) <= tool_names(ex):   # answer uses a tool that was not offered
                skipped += 1
                continue
            rows.append(ex)
        except Exception:
            skipped += 1

    rng = random.Random(SEED)
    rng.shuffle(rows)
    all_tools = sorted({n for ex in rows for n in tool_names(ex)})
    held_out = set(rng.sample(all_tools, int(HELD_OUT_FRAC * len(all_tools))))

    unseen_pool = [ex for ex in rows if gold_names(ex) <= held_out]         # needs only held-out tools
    seen_pool   = [ex for ex in rows if not (tool_names(ex) & held_out)]    # never shows a held-out tool

    test, _          = take(unseen_pool, *N_TEST)
    val, seen_rest   = take(seen_pool, *N_VAL)
    train, _         = take(seen_rest, *N_TRAIN)
    rng.shuffle(train)

    splits = {"train": train, "val": val, "test": test}
    for s, p in paths.items():
        write_jsonl(p, splits[s])
    with open(f"{DATA_DIR}/held_out_tools.json", "w") as f:
        json.dump(sorted(held_out), f)
    print(f"Kept {len(rows)} usable rows, skipped {skipped}.")
    print(f"{len(all_tools)} tools in total, {len(held_out)} held out.")

train, val, test = splits["train"], splits["val"], splits["test"]

# Leak check: no tool needed in the test set is ever shown during training.
train_tools = {n for ex in train + val for n in tool_names(ex)}
test_gold   = {n for ex in test for n in gold_names(ex)}
assert not (train_tools & test_gold), "Leak: a test tool appears in training data"

for name, split in splits.items():
    n_nc = sum(e["kind"] == "no_call" for e in split)
    print(f"{name:5s}: {len(split) - n_nc} call examples, {n_nc} no-suitable-tool examples")
for name, want in [("test", N_TEST), ("val", N_VAL), ("train", N_TRAIN)]:
    if len(splits[name]) < sum(want):
        print(f"WARNING: {name} has {len(splits[name])} examples, wanted {sum(want)}. Tell Claude.")

print("\nExample query:", test[0]["query"])
print("Gold answer:  ", test[0]["answers"])

Loaded existing splits from Drive.
train: 7200 call examples, 800 no-suitable-tool examples
val  : 270 call examples, 30 no-suitable-tool examples
test : 400 call examples, 150 no-suitable-tool examples

Example query: What are the auto-complete suggestions for stays in New York in English, French, and German?
Gold answer:   [{'name': 'stays_auto_complete', 'arguments': {'location': 'New York', 'language_code': 'en-us'}}, {'name': 'stays_auto_complete', 'arguments': {'location': 'New York', 'language_code': 'fr'}}, {'name': 'stays_auto_complete', 'arguments': {'location': 'New York', 'language_code': 'de'}}]


## 7. Parse model output and score it by code

- **format_ok** — the output was usable: a tool call that parsed as valid JSON (or, for no-suitable-tool examples, no call at all)
- **names_ok** — called exactly the right function(s)
- **exact** — right function(s) *and* right arguments (order of calls ignored; `"5"` and `5` treated as equal; strings compared case-insensitively). For no-suitable-tool examples, exact means the model made no call.

The last lines are a self-check: the correct answers, fed through the parser, must score 100%.

In [7]:
TOOL_CALL_RE = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)

def parse_tool_calls(text):
    # Returns (calls, format_ok). calls = [{"name":..., "arguments": {...}}]
    blocks = TOOL_CALL_RE.findall(text)
    if not blocks:  # model forgot the tags but may have written bare JSON
        s = text.strip()
        blocks = [s] if s.startswith(("{", "[")) else []
    calls, ok = [], bool(blocks)
    for b in blocks:
        try:
            obj = json.loads(b)
        except Exception:
            ok = False
            continue
        for o in (obj if isinstance(obj, list) else [obj]):
            if not (isinstance(o, dict) and "name" in o):
                ok = False
                continue
            args = o.get("arguments", {})
            if isinstance(args, str):
                try:
                    args = json.loads(args)
                except Exception:
                    ok, args = False, {}
            calls.append({"name": o["name"], "arguments": args or {}})
    return calls, ok and len(calls) > 0

def norm(v):
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return round(float(v), 6)
    if isinstance(v, str):
        s = v.strip()
        try:
            return round(float(s), 6)
        except ValueError:
            return s.lower()
    if isinstance(v, list):
        return [norm(x) for x in v]
    if isinstance(v, dict):
        return {str(k): norm(x) for k, x in sorted(v.items())}
    return v

def call_key(c):
    return c["name"] + "|" + json.dumps(norm(c.get("arguments") or {}), sort_keys=True)

def score(pred_calls, gold_calls, format_ok):
    if not gold_calls:                       # no suitable tool: correct means no call was made
        ok = len(pred_calls) == 0
        return {"format_ok": ok or format_ok, "names_ok": ok, "exact": ok}
    return {
        "format_ok": format_ok,
        "names_ok": sorted(c["name"] for c in pred_calls) == sorted(c["name"] for c in gold_calls),
        "exact": sorted(map(call_key, pred_calls)) == sorted(map(call_key, gold_calls)),
    }

def failure_type(r):
    if not r["gold"]:
        return "called a tool when none fit"
    if not r["pred"] and "tool_call" not in r["raw"]:
        return "answered in text instead of calling"
    if not r["format_ok"]:
        return "broken call format"
    if len(r["pred"]) != len(r["gold"]):
        return "wrong number of calls"
    if not r["names_ok"]:
        return "wrong function"
    return "right function, wrong arguments"

# Self-check: correct answers written in Qwen's format must score 100%.
for ex in test[:50] + test[-50:]:
    fake = "".join(f"<tool_call>\n{json.dumps(a)}\n</tool_call>\n" for a in ex["answers"]) \
           or "None of the available tools can do that."
    calls, ok = parse_tool_calls(fake)
    assert score(calls, ex["answers"], ok)["exact"], ex["id"]
print("Scorer self-check passed.")

Scorer self-check passed.


## 8. Evaluation runner (saves after every batch, resumes after disconnects)

In [8]:
def summarize(path, ids=None):
    rs = [r for r in read_jsonl(path) if ids is None or r["id"] in ids]
    if not rs:
        return {}
    pct = lambda xs: round(100 * sum(xs) / len(xs), 1) if xs else None
    call = [r for r in rs if r["kind"] == "call"]
    nc   = [r for r in rs if r["kind"] == "no_call"]
    return {"n_call": len(call),
            "unseen_tool_exact_%": pct([r["exact"] for r in call]),
            "right_function_%":    pct([r["names_ok"] for r in call]),
            "valid_format_%":      pct([r["format_ok"] for r in call]),
            "n_no_tool": len(nc),
            "no_tool_correct_%":   pct([r["exact"] for r in nc]),
            "sec_per_example": round(sum(r["sec_per_example"] for r in rs) / len(rs), 3)}

def run_eval(name, examples, generate_fn, batch_size=16):
    out_path = f"{RESULTS_DIR}/{name}.jsonl"
    done = {r["id"] for r in read_jsonl(out_path)}
    todo = [e for e in examples if e["id"] not in done]
    print(f"[{name}] {len(done)} already done, {len(todo)} to go")
    for i in range(0, len(todo), batch_size):
        batch = todo[i:i + batch_size]
        t0 = time.time()
        texts = generate_fn(batch)
        sec = (time.time() - t0) / len(batch)
        with open(out_path, "a") as f:
            for ex, text in zip(batch, texts):
                calls, ok = parse_tool_calls(text)
                rec = {"id": ex["id"], "kind": ex["kind"], "raw": text, "pred": calls,
                       "gold": ex["answers"], **score(calls, ex["answers"], ok),
                       "sec_per_example": sec}
                f.write(json.dumps(rec) + "\n")
        print(f"  {min(i + batch_size, len(todo))}/{len(todo)}  ({sec:.2f}s/example)")
    return summarize(out_path)

## 9. Load the small model

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="cuda").eval()

SYSTEM = ("You are a function-calling assistant. Answer the user's request by calling the provided tools. "
          "If none of the provided tools can do what the user asks, do not call any tool; "
          "reply in plain text saying so.")

def build_prompt(ex):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ex["query"]}]
    return tok.apply_chat_template(msgs, tools=ex["tools"], tokenize=False, add_generation_prompt=True)

print(build_prompt(test[0])[-1200:])   # look at the end of one prompt

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

You are a function-calling assistant. Answer the user's request by calling the provided tools. If none of the provided tools can do what the user asks, do not call any tool; reply in plain text saying so.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "stays_auto_complete", "description": "Fetches auto-complete suggestions for stays based on a given location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The location for which to get auto-complete suggestions. (default: ch)"}, "language_code": {"type": "string", "description": "The language code for the results. Defaults to 'en-us'. (default: en-us)"}}, "required": []}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>

## 10. Baseline: small model on all 550 test examples (~5–10 min on L4)

In [10]:
@torch.no_grad()
def hf_generate(batch, max_new_tokens=256):
    enc = tok([build_prompt(e) for e in batch], return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, top_k=None,
                         pad_token_id=tok.pad_token_id)
    # keep special tokens so <tool_call> tags survive, then strip end-of-turn markers
    texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=False)
    return [t.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip() for t in texts]

small_summary = run_eval("qwen1.5b_base", test, hf_generate, batch_size=16)
small_summary

[qwen1.5b_base] 550 already done, 0 to go


{'n_call': 400,
 'unseen_tool_exact_%': 80.0,
 'right_function_%': 98.2,
 'valid_format_%': 99.8,
 'n_no_tool': 150,
 'no_tool_correct_%': 76.0,
 'sec_per_example': 0.372}

## 11. Baseline: large model on 200 test examples (optional)

Uses Groq's OpenAI-compatible API. Skipped if there's no `GROQ_API_KEY`. If you hit rate limits it waits and retries; if it still fails it stops, and re-running the cell continues from where it stopped.

In [20]:
import copy

big_set = ([e for e in test if e["kind"] == "call"][:N_BIG[0]]
           + [e for e in test if e["kind"] == "no_call"][:N_BIG[1]])
BIG_RUN = "big_gptoss120b_multiturn"
MAX_TURNS = 8
TOOL_REPLY = "Call recorded. Make any remaining calls needed for the request, or reply DONE."

def relax_tools(tools):
    # The dataset marks some decimal fields (e.g. latitude) as integers. Groq enforces that
    # and rejects correct calls, so tell Groq these fields accept any number.
    tools = copy.deepcopy(tools)
    for t in tools:
        for spec in t["function"]["parameters"].get("properties", {}).values():
            if spec.get("type") == "integer":
                spec["type"] = "number"
            if spec.get("type") == "array" and spec.get("items", {}).get("type") == "integer":
                spec["items"]["type"] = "number"
    return tools

# Redo saved examples where the model made fewer calls than needed (cut short by a rejection).
big_path = f"{RESULTS_DIR}/{BIG_RUN}.jsonl"
saved = read_jsonl(big_path)
keep = [r for r in saved if not (r["gold"] and len(r["pred"]) < len(r["gold"]))]
if len(keep) < len(saved):
    write_jsonl(big_path, keep)
    print(f"Will redo {len(saved) - len(keep)} incomplete examples.")

if GROQ_API_KEY:
    from openai import OpenAI, BadRequestError, RateLimitError
    client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1", max_retries=0)
    rejected = {"count": 0, "shown": 0}
    limit_msg_shown = {"done": False}

    def recover_rejected_call(err):
        body = err.body if isinstance(err.body, dict) else {}
        info = body.get("error", body)
        fg = info.get("failed_generation") or ""
        rejected["count"] += 1
        if rejected["shown"] < 3:
            rejected["shown"] += 1
            print("    Groq rejected a tool call:", str(info.get("message"))[:600])
            print("    Model's attempt:", str(fg)[:400])
        m = re.search(r"<function=([\w\.\-]+)>?\s*(\{.*\})", fg, re.S)
        if m:
            fg = json.dumps({"name": m.group(1), "arguments": m.group(2)})
        try:
            obj = json.loads(fg)
            items = obj if isinstance(obj, list) else [obj]
            for o in items:
                if isinstance(o, dict) and "arguments" not in o and "parameters" in o:
                    o["arguments"] = o.pop("parameters")
            fg = json.dumps(items)
        except Exception:
            pass
        return f"<tool_call>\n{fg}\n</tool_call>\n"

    def wait_seconds(err_text, attempt):
        # Groq says how long to wait, e.g. "Please try again in 7m12.5s".
        m = re.search(r"try again in (?:(\d+)h)?(?:(\d+)m)?([\d\.]+)s", err_text)
        if m:
            return int(m.group(1) or 0) * 3600 + int(m.group(2) or 0) * 60 + float(m.group(3)) + 2
        return 10 * 2 ** attempt

    def ask(messages, tools):
        for attempt in range(6):
            try:
                return client.chat.completions.create(
                    model=BIG_MODEL, messages=messages, tools=tools,
                    tool_choice="auto", temperature=0), None
            except BadRequestError as e:
                return None, recover_rejected_call(e)
            except RateLimitError as e:
                text = str(e)
                if not limit_msg_shown["done"]:
                    limit_msg_shown["done"] = True
                    print("    FULL RATE LIMIT MESSAGE:", text[:900])
                wait = wait_seconds(text, attempt)
                if wait > 600:
                    raise RuntimeError(f"Groq limit reached; it asks to wait {wait/60:.0f} minutes. "
                                       "Re-run this cell after that to resume.")
                print(f"    Rate limit; waiting {wait:.0f}s")
                time.sleep(wait)
            except Exception as e:
                wait = 5 * 2 ** attempt
                print(f"    API error ({type(e).__name__}: {str(e)[:120]}); retrying in {wait}s")
                time.sleep(wait)
        raise RuntimeError("Groq API kept failing - re-run this cell later to resume.")

    def big_generate(batch):
        texts = []
        for ex in batch:
            tools = relax_tools(ex["tools"])
            messages = [{"role": "system", "content": SYSTEM},
                        {"role": "user", "content": ex["query"]}]
            out, seen, last_text = "", set(), ""
            for turn in range(MAX_TURNS):
                r, recovered = ask(messages, tools)
                if recovered is not None:
                    out += recovered
                    break
                msg = r.choices[0].message
                if not msg.tool_calls:
                    last_text = msg.content or ""
                    break
                new_calls = 0
                for tc in msg.tool_calls:
                    key = tc.function.name + "|" + (tc.function.arguments or "")
                    if key in seen:
                        continue
                    seen.add(key)
                    new_calls += 1
                    try:
                        args = json.loads(tc.function.arguments or "{}")
                    except Exception:
                        args = {"__unparsed__": tc.function.arguments}
                    out += f"<tool_call>\n{json.dumps({'name': tc.function.name, 'arguments': args})}\n</tool_call>\n"
                if new_calls == 0:
                    break
                messages.append({"role": "assistant", "content": msg.content or "",
                                 "tool_calls": [{"id": tc.id, "type": "function",
                                                 "function": {"name": tc.function.name,
                                                              "arguments": tc.function.arguments or "{}"}}
                                                for tc in msg.tool_calls]})
                for tc in msg.tool_calls:
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": TOOL_REPLY})
            texts.append(out or ("TEXT: " + last_text))
        return texts

    big_summary = run_eval(BIG_RUN, big_set, big_generate, batch_size=5)
    print(big_summary)
    print(f"Groq rejected {rejected['count']} tool calls in this run.")
else:
    print("Skipped (no GROQ_API_KEY).")

Will redo 5 incomplete examples.
[big_gptoss120b_multiturn] 78 already done, 122 to go
    FULL RATE LIMIT MESSAGE: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m3ymp2f6e7mbcscz6g94f8xc` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199491, Requested 589. Please try again in 34.56s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
    Rate limit; waiting 37s


KeyboardInterrupt: 

In [11]:
import gc, subprocess, sys

BIG_LOCAL_ID = "Qwen/Qwen2.5-7B-Instruct"
BIG_LOCAL_RUN = "qwen7b_base"

already = {r["id"] for r in read_jsonl(f"{RESULTS_DIR}/{BIG_LOCAL_RUN}.jsonl")}
if all(e["id"] in already for e in test):
    print("Qwen2.5-7B baseline already complete.")
else:
    # Free the small model's memory first (its results are already saved).
    try:
        del model
    except NameError:
        pass
    gc.collect()
    torch.cuda.empty_cache()

    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    big_tok = AutoTokenizer.from_pretrained(BIG_LOCAL_ID)
    big_tok.padding_side = "left"
    if vram_gb >= 20:      # L4 / A100: load at full precision
        big_model = AutoModelForCausalLM.from_pretrained(
            BIG_LOCAL_ID, torch_dtype=dtype, device_map="cuda").eval()
    else:                  # T4: load in 4-bit so it fits in 16 GB
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=True)
        from transformers import BitsAndBytesConfig
        big_model = AutoModelForCausalLM.from_pretrained(
            BIG_LOCAL_ID,
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
            device_map="cuda").eval()
    print(f"Loaded {BIG_LOCAL_ID} on a {vram_gb:.0f} GB GPU")

    def build_prompt_big(ex):
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ex["query"]}]
        return big_tok.apply_chat_template(msgs, tools=ex["tools"], tokenize=False, add_generation_prompt=True)

    @torch.no_grad()
    def big_local_generate(batch, max_new_tokens=256):
        enc = big_tok([build_prompt_big(e) for e in batch], return_tensors="pt", padding=True).to(big_model.device)
        out = big_model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 temperature=None, top_p=None, top_k=None,
                                 pad_token_id=big_tok.pad_token_id)
        texts = big_tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=False)
        return [t.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip() for t in texts]

    print(run_eval(BIG_LOCAL_RUN, test, big_local_generate, batch_size=8))

Qwen2.5-7B baseline already complete.


## 12. Results table (both models compared on the same 200 examples) + failure categories

In [12]:
import pandas as pd
from collections import Counter

DEFAULT_RE = re.compile(r"\(default: (.*)\)\s*$", re.S)
id2ex = {e["id"]: e for e in test}

def tool_defaults(ex):
    out = {}
    for t in ex["tools"]:
        props = t["function"]["parameters"].get("properties", {})
        out[t["function"]["name"]] = {}
        for p, spec in props.items():
            m = DEFAULT_RE.search(spec.get("description", ""))
            if m:
                out[t["function"]["name"]][p] = m.group(1)
    return out

def clean_call(call, defaults):
    # Drop arguments that are empty or equal to the tool's default: they do not change the call.
    d = defaults.get(call["name"], {})
    args = call.get("arguments") if isinstance(call.get("arguments"), dict) else {}
    kept = {}
    for k, v in args.items():
        if v is None or v == "":
            continue
        if k in d and (norm(v) == norm(d[k]) or str(v).strip().lower() == d[k].strip().lower()):
            continue
        kept[k] = v
    return {"name": call["name"], "arguments": kept}

def lenient_exact(r):
    if not r["gold"]:
        return r["exact"]
    d = tool_defaults(id2ex[r["id"]])
    return (sorted(call_key(clean_call(c, d)) for c in r["pred"])
            == sorted(call_key(clean_call(c, d)) for c in r["gold"]))

def pct(xs):
    return round(100 * sum(xs) / len(xs), 1) if xs else None

def row(label, rs):
    call = [r for r in rs if r["kind"] == "call"]
    nc = [r for r in rs if r["kind"] == "no_call"]
    one = [r for r in call if len(r["gold"]) == 1]
    multi = [r for r in call if len(r["gold"]) >= 2]
    return {"model": label, "n_call": len(call),
            "exact_strict_%": pct([r["exact"] for r in call]),
            "exact_fair_%": pct([lenient_exact(r) for r in call]),
            "1_call_fair_%": pct([lenient_exact(r) for r in one]),
            "2+_calls_fair_%": pct([lenient_exact(r) for r in multi]),
            "right_function_%": pct([r["names_ok"] for r in call]),
            "n_no_tool": len(nc), "no_tool_correct_%": pct([r["exact"] for r in nc]),
            "sec_per_example": round(sum(r["sec_per_example"] for r in rs) / len(rs), 3)}

small_rs = read_jsonl(f"{RESULTS_DIR}/qwen1.5b_base.jsonl")
q7_rs    = read_jsonl(f"{RESULTS_DIR}/qwen7b_base.jsonl")
oss_rs   = read_jsonl(f"{RESULTS_DIR}/big_gptoss120b_multiturn.jsonl")

table = [row("Qwen2.5-1.5B (all 550)", small_rs)]
if q7_rs:
    table.append(row(f"Qwen2.5-7B ({len(q7_rs)} done)", q7_rs))
if oss_rs:   # partial run: compare all three on exactly the examples gpt-oss has finished
    ids = {r["id"] for r in oss_rs}
    table.append(row(f"Qwen2.5-1.5B (same {len(ids)})", [r for r in small_rs if r["id"] in ids]))
    if q7_rs:
        table.append(row(f"Qwen2.5-7B (same {len(ids)})", [r for r in q7_rs if r["id"] in ids]))
    table.append(row(f"gpt-oss-120b multi-turn ({len(ids)}, partial)", oss_rs))
results_df = pd.DataFrame(table)
display(results_df)

failure_counts = {}
for name, rs in [("qwen1.5b_base", small_rs), ("qwen7b_base", q7_rs)]:
    fails = [r for r in rs if not lenient_exact(r)]
    if rs:
        failure_counts[name] = dict(Counter(map(failure_type, fails)))
        print(f"\n{name}: {len(fails)} failures out of {len(rs)} (fair scoring)")
        for k, v in Counter(map(failure_type, fails)).most_common():
            print(f"  {v:4d}  {k}")

with open(f"{RESULTS_DIR}/stage1_summary.json", "w") as f:
    json.dump({"results": table, "failures": failure_counts}, f, indent=2)
print("\nSaved", f"{RESULTS_DIR}/stage1_summary.json")

,model,n_call,exact_strict_%,exact_fair_%,1_call_fair_%,2+_calls_fair_%,right_function_%,n_no_tool,no_tool_correct_%,sec_per_example
0,Qwen2.5-1.5B (all 550),400,80.0,84.5,87.2,79.3,98.2,150,76.0,0.372
1,Qwen2.5-7B (550 done),400,84.8,87.5,90.9,80.7,98.8,150,87.3,1.252
2,Qwen2.5-1.5B (same 78),78,78.2,83.3,88.1,77.8,97.4,0,NaN,0.451
3,Qwen2.5-7B (same 78),78,82.1,84.6,90.5,77.8,98.7,0,NaN,1.055
4,"gpt-oss-120b multi-turn (78, partial)",78,78.2,80.8,73.8,88.9,88.5,0,NaN,8.546



qwen1.5b_base: 98 failures out of 550 (fair scoring)
    55  right function, wrong arguments
    36  called a tool when none fit
     5  wrong number of calls
     1  wrong function
     1  answered in text instead of calling

qwen7b_base: 69 failures out of 550 (fair scoring)
    45  right function, wrong arguments
    19  called a tool when none fit
     4  wrong number of calls
     1  answered in text instead of calling

Saved /content/drive/MyDrive/smallcall/results_v2/stage1_summary.json


## 13. Read a few small-model failures yourself

In [14]:
id2q = {e["id"]: e["query"] for e in test}
fails = [r for r in read_jsonl(f"{RESULTS_DIR}/qwen1.5b_base.jsonl") if not r["exact"]]
for r in random.Random(0).sample(fails, min(6, len(fails))):
    print("=" * 100)
    print("TYPE :", failure_type(r))
    print("QUERY:", id2q[r["id"]])
    print("GOLD :", json.dumps(r["gold"]) if r["gold"] else "(no call expected)")
    print("PRED :", json.dumps(r["pred"]))
    print("RAW  :", r["raw"][:400])

TYPE : called a tool when none fit
QUERY: Fetch detailed photos of a property in Costa Rica with the ID 'cr/la-buena-vida-cabinas' and display the results in English.
GOLD : (no call expected)
PRED : [{"name": "webcam_search", "arguments": {"query": "cr/la-buena-vida-cabinas"}}]
RAW  : <tool_call>
{"name": "webcam_search", "arguments": {"query": "cr/la-buena-vida-cabinas"}}
</tool_call>
TYPE : right function, wrong arguments
QUERY: Could you fetch the trending music on Spotify's home page for the UK region?
GOLD : [{"name": "get_home_page_overview", "arguments": {"region": "UK"}}]
PRED : [{"name": "get_home_page_overview", "arguments": {"region": "GB"}}]
RAW  : <tool_call>
{"name": "get_home_page_overview", "arguments": {"region": "GB"}}
</tool_call>
TYPE : called a tool when none fit
QUERY: Fetch a subset list of the 'weather_data' dataset along with its metadata in JSON format from the ODAM API.
GOLD : (no call expected)
PRED : [{"name": "tsv1", "arguments": {"dataset": "weather_data

## Stage 1 done

Send Claude:
1. The table from cell 12
2. The failure counts printed under it
3. Anything in cell 13 where the **scorer** looks wrong rather than the model — for example the prediction looks correct but is marked as a failure, or a "no call expected" example where one of the remaining tools really could answer the question

Next is **Stage 2: SFT with LoRA** on the 8,000 training examples already saved in your Drive.

In [9]:
from collections import Counter

big = read_jsonl(f"{RESULTS_DIR}/big_gptoss120b.jsonl")
small = read_jsonl(f"{RESULTS_DIR}/qwen1.5b_base.jsonl")
id2q = {e["id"]: e["query"] for e in test}

print("BIG MODEL: (calls expected, calls made) for 'wrong number of calls' failures")
wrong_n = [r for r in big if not r["exact"] and r["gold"] and r["pred"] and len(r["pred"]) != len(r["gold"])]
for (g, p), n in Counter((len(r["gold"]), len(r["pred"])) for r in wrong_n).most_common():
    print(f"  expected {g}, made {p}: {n} cases")

print("\nBIG MODEL: accuracy split by how many calls the question needs")
for label, keep in [("needs 1 call", lambda r: len(r["gold"]) == 1), ("needs 2+ calls", lambda r: len(r["gold"]) >= 2)]:
    rs = [r for r in big if keep(r)]
    if rs:
        print(f"  {label}: {sum(r['exact'] for r in rs)}/{len(rs)} correct")

print("\nSMALL MODEL: same split, on the same 200 examples")
big_ids = {r["id"] for r in big}
for label, keep in [("needs 1 call", lambda r: len(r["gold"]) == 1), ("needs 2+ calls", lambda r: len(r["gold"]) >= 2)]:
    rs = [r for r in small if r["id"] in big_ids and keep(r)]
    if rs:
        print(f"  {label}: {sum(r['exact'] for r in rs)}/{len(rs)} correct")

print("\nBIG MODEL: 3 examples of 'wrong number of calls'")
for r in wrong_n[:3]:
    print("-" * 90)
    print("QUERY:", id2q[r["id"]][:300])
    print("GOLD :", json.dumps(r["gold"])[:400])
    print("PRED :", json.dumps(r["pred"])[:400])

print("\nSMALL MODEL: 8 examples of 'right function, wrong arguments'")
wa = [r for r in small if not r["exact"] and failure_type(r) == "right function, wrong arguments"]
for r in random.Random(1).sample(wa, min(8, len(wa))):
    print("-" * 90)
    print("QUERY:", id2q[r["id"]][:300])
    print("GOLD :", json.dumps(r["gold"])[:400])
    print("PRED :", json.dumps(r["pred"])[:400])

BIG MODEL: (calls expected, calls made) for 'wrong number of calls' failures
  expected 2, made 1: 48 cases
  expected 3, made 1: 9 cases
  expected 4, made 1: 2 cases
  expected 6, made 1: 1 cases

BIG MODEL: accuracy split by how many calls the question needs
  needs 1 call: 76/88 correct
  needs 2+ calls: 0/62 correct

SMALL MODEL: same split, on the same 200 examples
  needs 1 call: 71/88 correct
  needs 2+ calls: 44/62 correct

BIG MODEL: 3 examples of 'wrong number of calls'
------------------------------------------------------------------------------------------
QUERY: What are the auto-complete suggestions for stays in New York in English, French, and German?
GOLD : [{"name": "stays_auto_complete", "arguments": {"location": "New York", "language_code": "en-us"}}, {"name": "stays_auto_complete", "arguments": {"location": "New York", "language_code": "fr"}}, {"name": "stays_auto_complete", "arguments": {"location": "New York", "language_code": "de"}}]
PRED : [{"name": "stays_aut

Code part 2

In [13]:
# part 2 cell 1: setup for training
import gc, subprocess, sys, math

needed = ["train", "val", "test", "run_eval", "read_jsonl", "parse_tool_calls", "score", "SYSTEM",
          "MODEL_ID", "RESULTS_DIR", "PROJECT_DIR", "SEED", "lenient_exact", "row", "failure_type"]
missing = [n for n in needed if n not in globals()]
assert not missing, f"Run the Part 1 cells first (3 to 9, and 12). Missing: {missing}"

# Free GPU memory held by earlier models (their results are saved).
for name in ["big_model", "model", "base"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)

RUN_TAG = "sft_v1"                       # change this if you retrain with different settings
ADAPTER_DIR = f"{PROJECT_DIR}/adapters/{RUN_TAG}"
EPOCHS, LR, BATCH, ACCUM = 2, 1e-4, 8, 2  # 2 passes over the data; effective batch size 16
MAX_LEN, LORA_R = 1024, 16

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"

def build_prompt(ex):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ex["query"]}]
    return tok.apply_chat_template(msgs, tools=ex["tools"], tokenize=False, add_generation_prompt=True)

def make_generate(m):
    @torch.no_grad()
    def gen(batch, max_new_tokens=256):
        m.eval()
        enc = tok([build_prompt(e) for e in batch], return_tensors="pt", padding=True).to(m.device)
        out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=False)
        return [t.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip() for t in texts]
    return gen

id2ex = {e["id"]: e for e in test + val}   # fair scoring needs the tool definitions for val too
print(f"Ready. GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

Ready. GPU memory free: 84.6 GB


In [14]:
# part 2 cell 2: build the training examples
NO_TOOL_REPLY = "None of the available tools can do this, so I can't complete the request."

def target_text(ex):
    # What the model should write: the tool calls, or a plain refusal when no tool fits.
    if not ex["answers"]:
        return NO_TOOL_REPLY
    return "\n".join(
        "<tool_call>\n" + json.dumps({"name": a["name"], "arguments": a["arguments"]}, ensure_ascii=False)
        + "\n</tool_call>" for a in ex["answers"])

def encode(ex):
    # The model is trained only on the answer part; the prompt part is masked with -100.
    p = tok(build_prompt(ex), add_special_tokens=False)["input_ids"]
    t = tok(target_text(ex) + "<|im_end|>", add_special_tokens=False)["input_ids"]
    return p + t, [-100] * len(p) + t

# Check: every training answer, read back by our parser, scores as correct.
for ex in train[:300]:
    calls, ok = parse_tool_calls(target_text(ex))
    assert score(calls, ex["answers"], ok)["exact"], ex["id"]

encoded = [encode(e) for e in train]
train_enc = [x for x in encoded if len(x[0]) <= MAX_LEN]
lengths = sorted(len(x[0]) for x in train_enc)
n_no_tool = sum(1 for e in train if not e["answers"])
print(f"{len(train_enc)} training examples kept ({len(encoded) - len(train_enc)} too long were dropped)")
print(f"{n_no_tool} of {len(train)} are no-suitable-tool examples")
print(f"Length in tokens: median {lengths[len(lengths) // 2]}, longest {lengths[-1]}")
print("\nEnd of one training example:\n", tok.decode(train_enc[0][0][-120:]))

7836 training examples kept (164 too long were dropped)
800 of 8000 are no-suitable-tool examples
Length in tokens: median 468, longest 1024

End of one training example:
  US)"}}, "required": []}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
What is the market capitalization of Bitcoin and Ethereum as of today?<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "market_capitalization", "arguments": {"symbol": "BTC"}}
</tool_call>
<tool_call>
{"name": "market_capitalization", "arguments": {"symbol": "ETH"}}
</tool_call><|im_end|>


In [15]:
# part 2 cell 2b: remove the outdated torchao package that blocks peft (we don't use it)
import subprocess, sys, importlib

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
for name in [m for m in sys.modules if m == "torchao" or m.startswith("torchao.")]:
    del sys.modules[name]
importlib.invalidate_caches()

import importlib.util
print("torchao still present:", importlib.util.find_spec("torchao") is not None)

torchao still present: False


In [16]:
# part 2 cell 2c: use smaller batches and release the memory held by the failed run
import sys, gc

BATCH, ACCUM = 4, 4          # 4 examples at a time, update every 4 batches (effective batch 16)

# After an out-of-memory error, the notebook keeps the failed run's data alive. Let go of it.
sys.last_traceback = sys.last_value = sys.last_type = None
try:
    sys.last_exc = None
except Exception:
    pass
for name in ["model", "base", "opt", "sched", "loss", "inp", "att", "lab"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f"GPU memory free: {free_gb:.1f} GB")
print("OK to run part 2 cell 3." if free_gb > 15 else
      "Memory was not released. Use Runtime -> Restart session, then rerun Part 1 cells 3-9 and 12, "
      "Part 2 cells 1 and 2, this cell, and then cell 3.")

GPU memory free: 84.6 GB
OK to run part 2 cell 3.


In [17]:
# part 2 cell 3: train with LoRA (saves after each epoch; rerun to resume)
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import get_cosine_schedule_with_warmup

for name in ["model", "base"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

os.makedirs(ADAPTER_DIR, exist_ok=True)
done_epochs = max([int(d[5:]) for d in os.listdir(ADAPTER_DIR)
                   if d.startswith("epoch") and os.path.exists(f"{ADAPTER_DIR}/{d}/adapter_config.json")] or [0])

train_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=train_dtype, device_map="cuda")
if done_epochs:
    print(f"Resuming after epoch {done_epochs}")
    model = PeftModel.from_pretrained(base, f"{ADAPTER_DIR}/epoch{done_epochs}", is_trainable=True)
else:
    model = get_peft_model(base, LoraConfig(
        r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

def make_batches(epoch):
    idx = list(range(len(train_enc)))
    random.Random(SEED + epoch).shuffle(idx)
    return [idx[i:i + BATCH] for i in range(0, len(idx), BATCH)]

def collate(ids):
    items = [train_enc[i] for i in ids]
    L = max(len(x[0]) for x in items)
    inp = torch.full((len(items), L), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(items), L), -100, dtype=torch.long)
    att = torch.zeros((len(items), L), dtype=torch.long)
    for j, (x, y) in enumerate(items):
        inp[j, :len(x)] = torch.tensor(x)
        lab[j, :len(y)] = torch.tensor(y)
        att[j, :len(x)] = 1
    return inp.cuda(), att.cuda(), lab.cuda()

steps_per_epoch = math.ceil(len(make_batches(1)) / ACCUM)
total_steps = EPOCHS * steps_per_epoch
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
sched = get_cosine_schedule_with_warmup(opt, int(0.05 * total_steps), total_steps)
for _ in range(done_epochs * steps_per_epoch):     # put the learning-rate schedule back where it was
    sched.step()

for epoch in range(done_epochs + 1, EPOCHS + 1):
    model.train()
    batches, running, t0 = make_batches(epoch), [], time.time()
    for b, ids in enumerate(batches, 1):
        inp, att, lab = collate(ids)
        loss = model(input_ids=inp, attention_mask=att, labels=lab).loss
        if not math.isfinite(loss.item()):
            raise RuntimeError(f"Loss became {loss.item()} at batch {b}. Tell Claude.")
        (loss / ACCUM).backward()
        running.append(loss.item())
        if b % ACCUM == 0 or b == len(batches):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)
        if b % 100 == 0:
            print(f"epoch {epoch}  batch {b}/{len(batches)}  loss {sum(running[-100:]) / 100:.4f}  "
                  f"({(time.time() - t0) / 60:.1f} min)")
    model.save_pretrained(f"{ADAPTER_DIR}/epoch{epoch}")
    print(f"Saved epoch {epoch}. Checking it on the validation set...")
    run_eval(f"{RUN_TAG}_e{epoch}_val", val, make_generate(model), batch_size=16)
    r = row("", read_jsonl(f"{RESULTS_DIR}/{RUN_TAG}_e{epoch}_val.jsonl"))
    print(f"VALIDATION epoch {epoch}: correct call {r['exact_fair_%']}%   no-tool correct {r['no_tool_correct_%']}%")

print("Training finished.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
epoch 1  batch 100/1959  loss 0.1221  (0.6 min)
epoch 1  batch 200/1959  loss 0.0479  (1.1 min)
epoch 1  batch 300/1959  loss 0.0520  (1.6 min)
epoch 1  batch 400/1959  loss 0.0296  (2.2 min)
epoch 1  batch 500/1959  loss 0.0425  (2.7 min)
epoch 1  batch 600/1959  loss 0.0254  (3.2 min)
epoch 1  batch 700/1959  loss 0.0282  (3.8 min)
epoch 1  batch 800/1959  loss 0.0273  (4.3 min)
epoch 1  batch 900/1959  loss 0.0248  (4.8 min)
epoch 1  batch 1000/1959  loss 0.0226  (5.4 min)
epoch 1  batch 1100/1959  loss 0.0283  (5.9 min)
epoch 1  batch 1200/1959  loss 0.0290  (6.4 min)
epoch 1  batch 1300/1959  loss 0.0265  (6.9 min)
epoch 1  batch 1400/1959  loss 0.0417  (7.5 min)
epoch 1  batch 1500/1959  loss 0.0380  (8.0 min)
epoch 1  batch 1600/1959  loss 0.0326  (8.5 min)
epoch 1  batch 1700/1959  loss 0.0278  (9.0 min)
epoch 1  batch 1800/1959  loss 0.0323  (9.5 min)
epoch 1  batch 1900/1959  loss 0.0350  (10.1 mi

In [18]:
# part 2 cell 4: pick the best epoch, test it, compare with the baselines
import pandas as pd
from collections import Counter
from peft import PeftModel

def val_score(ep):
    rs = read_jsonl(f"{RESULTS_DIR}/{RUN_TAG}_e{ep}_val.jsonl")
    if len(rs) < len(val):
        return None
    r = row("", rs)
    return (r["exact_fair_%"] + r["no_tool_correct_%"]) / 2   # both skills count equally

scores = {ep: val_score(ep) for ep in range(1, EPOCHS + 1)}
valid = {k: v for k, v in scores.items() if v is not None}
best = max(valid, key=valid.get) if valid else EPOCHS
print("Validation score per epoch:", scores, "-> using epoch", best)

for name in ["model", "base"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dt, device_map="cuda")
model = PeftModel.from_pretrained(base, f"{ADAPTER_DIR}/epoch{best}").eval()

TEST_RUN = f"{RUN_TAG}_e{best}_test"
run_eval(TEST_RUN, test, make_generate(model), batch_size=16)

base_rs = read_jsonl(f"{RESULTS_DIR}/qwen1.5b_base.jsonl")
sft_rs  = read_jsonl(f"{RESULTS_DIR}/{TEST_RUN}.jsonl")
q7_rs   = read_jsonl(f"{RESULTS_DIR}/qwen7b_base.jsonl")
table = [row("Qwen2.5-1.5B before training", base_rs),
         row(f"Qwen2.5-1.5B after SFT (epoch {best})", sft_rs)]
if q7_rs:
    table.append(row("Qwen2.5-7B (reference)", q7_rs))
display(pd.DataFrame(table))

# Same question, before vs after: what training fixed and what it broke.
before = {r["id"]: lenient_exact(r) for r in base_rs}
print("\nChanges on the same test questions:")
for kind, label in [("call", "questions needing a call"), ("no_call", "no-suitable-tool questions")]:
    rs = [r for r in sft_rs if r["kind"] == kind]
    fixed = sum(1 for r in rs if lenient_exact(r) and not before[r["id"]])
    broke = sum(1 for r in rs if not lenient_exact(r) and before[r["id"]])
    print(f"  {label}: {fixed} fixed, {broke} newly wrong (out of {len(rs)})")

fails = [r for r in sft_rs if not lenient_exact(r)]
print(f"\nAfter SFT: {len(fails)} failures out of {len(sft_rs)} (fair scoring)")
for k, v in Counter(map(failure_type, fails)).most_common():
    print(f"  {v:4d}  {k}")

with open(f"{RESULTS_DIR}/stage2_summary.json", "w") as f:
    json.dump({"best_epoch": best, "val_scores": scores, "results": table,
               "failures_after_sft": dict(Counter(map(failure_type, fails)))}, f, indent=2)
print("\nSaved", f"{RESULTS_DIR}/stage2_summary.json")

Validation score per epoch: {1: 90.75, 2: 92.05} -> using epoch 2


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[sft_v1_e2_test] 0 already done, 550 to go
  16/550  (0.50s/example)
  32/550  (0.69s/example)
  48/550  (0.80s/example)
  64/550  (1.34s/example)
  80/550  (0.51s/example)
  96/550  (0.50s/example)
  112/550  (0.79s/example)
  128/550  (0.43s/example)
  144/550  (0.38s/example)
  160/550  (0.57s/example)
  176/550  (0.46s/example)
  192/550  (0.53s/example)
  208/550  (0.39s/example)
  224/550  (0.46s/example)
  240/550  (0.72s/example)
  256/550  (0.77s/example)
  272/550  (0.40s/example)
  288/550  (0.45s/example)
  304/550  (0.52s/example)
  320/550  (0.64s/example)
  336/550  (0.44s/example)
  352/550  (0.65s/example)
  368/550  (0.43s/example)
  384/550  (0.42s/example)
  400/550  (0.62s/example)
  416/550  (0.13s/example)
  432/550  (0.33s/example)
  448/550  (0.13s/example)
  464/550  (0.17s/example)
  480/550  (0.12s/example)
  496/550  (0.13s/example)
  512/550  (0.11s/example)
  528/550  (0.34s/example)
  544/550  (0.13s/example)
  550/550  (0.27s/example)


,model,n_call,exact_strict_%,exact_fair_%,1_call_fair_%,2+_calls_fair_%,right_function_%,n_no_tool,no_tool_correct_%,sec_per_example
0,Qwen2.5-1.5B before training,400,80.0,84.5,87.2,79.3,98.2,150,76.0,0.372
1,Qwen2.5-1.5B after SFT (epoch 2),400,82.0,85.2,87.9,80.0,96.8,150,96.7,0.468
2,Qwen2.5-7B (reference),400,84.8,87.5,90.9,80.7,98.8,150,87.3,1.252



Changes on the same test questions:
  questions needing a call: 13 fixed, 10 newly wrong (out of 400)
  no-suitable-tool questions: 31 fixed, 0 newly wrong (out of 150)

After SFT: 64 failures out of 550 (fair scoring)
    46  right function, wrong arguments
     6  answered in text instead of calling
     5  wrong number of calls
     5  called a tool when none fit
     1  broken call format
     1  wrong function

Saved /content/drive/MyDrive/smallcall/results_v2/stage2_summary.json
